# HW2 — Deformable-DETR: exploration & visualization

Интерактивный ноутбук для разведки датасета, визуализации предсказаний и графиков ошибок.  
Полная пайплайн-логика (обучение, error-analysis) лежит в `src/`, ноутбук — про **визуализацию результатов**.

## Структура
1. Загрузка subset-аннотаций и просмотр распределения классов.
2. Визуализация GT-боксов на нескольких картинках.
3. Загрузка чекпойнта и предсказания на val.
4. Графики loss / mAP из TensorBoard event-файлов.
5. Разбор ошибок: примеры CLS / LOC / BKG / DUPE.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

## 1. Распределение классов в subset

In [ ]:
SUBSET = Path('../data/coco_subset')
ann = json.load(open(SUBSET / 'annotations' / 'instances_train_subset.json'))

cat_id_to_name = {c['id']: c['name'] for c in ann['categories']}
from collections import Counter
counts = Counter(a['category_id'] for a in ann['annotations'])
names  = [cat_id_to_name[i] for i in sorted(counts)]
vals   = [counts[i]            for i in sorted(counts)]

plt.figure(figsize=(9, 4))
plt.bar(names, vals)
plt.xticks(rotation=30, ha='right')
plt.ylabel('# instances')
plt.title('Class distribution (train subset)')
plt.tight_layout()
plt.show()

## 2. Визуализация GT

In [ ]:
from src.eval.visualize import draw_boxes

imgs = {i['id']: i for i in ann['images']}
anns_by_img = {}
for a in ann['annotations']:
    anns_by_img.setdefault(a['image_id'], []).append(a)

for img_id in list(imgs)[:4]:
    info = imgs[img_id]
    pil = Image.open(SUBSET / 'images' / 'train' / info['file_name']).convert('RGB')
    boxes_xyxy = []
    labels = []
    names_ = []
    for a in anns_by_img.get(img_id, []):
        x, y, w, h = a['bbox']
        boxes_xyxy.append([x, y, x + w, y + h])
        labels.append(a['category_id'])
        names_.append(cat_id_to_name[a['category_id']])
    vis = draw_boxes(pil, np.array(boxes_xyxy), labels, names_, color=(0, 200, 0))
    plt.figure(figsize=(6, 6))
    plt.imshow(vis); plt.axis('off'); plt.show()

## 3. Загрузка чекпойнта и инференс

Запускается после `scripts/run_detr.sh`.

In [ ]:
import torch
from src.models.detr_wrapper import build_model_and_processor

CKPT = '../checkpoints/detr/best.pt'
model, processor = build_model_and_processor('SenseTime/deformable-detr', num_labels=10)
state = torch.load(CKPT, map_location='cpu')
model.load_state_dict(state['model'])
model.eval();

## 4. Кривые из TensorBoard

Если хочется без TB — читаем event-файлы напрямую.

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

acc = EventAccumulator('../runs/detr')
acc.Reload()
for tag in ('train/loss', 'val/mAP', 'val/mAP_50'):
    if tag in acc.Tags()['scalars']:
        evs = acc.Scalars(tag)
        plt.plot([e.step for e in evs], [e.value for e in evs], label=tag)
plt.legend(); plt.grid(); plt.show()

## 5. Error-analysis сводка

После `scripts/run_error_analysis.sh` смотрим картинки:

In [ ]:
for p in sorted(Path('../reports/error_analysis').glob('*.png'))[:6]:
    plt.figure(figsize=(7, 7))
    plt.imshow(Image.open(p)); plt.axis('off'); plt.title(p.name); plt.show()